# CORE 检查点复评：review / explore

_只评估可信的本地最佳检查点，不训练、不选择超参数、不覆盖历史结果。_

---

- `review`：真实下一商品出现在当前模型可见的非 padding session 中。
- `explore`：真实下一商品不在上述 session 中。

两组使用相同的全商品排序，只排除 ID 0，不屏蔽历史商品。历史以模型收到的截断后输入为准，当前上限为 50。

请选择项目 `.venv` 的 Python kernel，并从上到下运行。缺少 Notebook 环境时，可在项目终端安装 `requirements_notebook.txt`；评估 LLM 还需要 `requirements_llm.txt` 和已有模型缓存。检查点采用 pickle 格式，只能加载可信文件。


## ⚙️ 环境与项目路径

默认从当前目录及其父目录寻找项目根目录；从项目根目录或其子目录启动 Notebook 即可。如果在其他位置启动，请先将下面的 `PROJECT_ROOT` 改为项目绝对路径。


In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'train.py').is_file() and (path / 'configs' / 'common.yaml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('请从 CORE 项目目录启动 Notebook，或手动指定 PROJECT_ROOT。')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from recbole.data.utils import get_dataloader
from recbole.utils import init_logger
from train import ROOT, _file_hash, _review_explore_report, _write_json, load_experiment
from trainer import CORETrainer

if ROOT.resolve() != PROJECT_ROOT.resolve():
    raise RuntimeError('当前 kernel 已加载其他项目的 train 模块，请重启 kernel。')
print(f'项目目录：{ROOT}')
print(f'Python kernel：{sys.executable}')
print(f'CUDA 可用：{torch.cuda.is_available()}')


项目目录：D:\DataSciencePractise\CORE
Python kernel：D:\DataSciencePractise\CORE\.venv\Scripts\python.exe
CUDA 可用：True


## ⚙️ 复评参数

只评估 `RUN_DIRECTORY` 指定的一组检查点；不会自动依次评估三个模型。

- `SPLIT='valid'`：默认验证集；最终方案固定后才显式改为 `'test'`。
- `LIMIT=None`：所选 split 全量评估；`LIMIT=128` 只检查前 128 条，标记为 `prefix_subset`，不能当作全量效果。
- `DEVICE`：可选 `'cuda'` 或 `'cpu'`；CPU 更慢且可能与 GPU 的数值略有差异。
- `EVAL_BATCH_SIZE=32`：沿用当前匹配实验的评价 batch。

更换模型时，修改 `RUN_DIRECTORY = RUN_DIRECTORIES['trm']` 或 `['llm']`，然后重新运行执行单元格。其他新实验可直接填入运行目录的路径。


In [2]:
RUN_DIRECTORIES = {
    'ave': ROOT / 'results' / 'phase1' / '20261005-115750-tmall-ave-9cf3dc',
    'trm': ROOT / 'results' / 'phase1' / '20261005-120616-tmall-trm-32ec4d',
    'llm': ROOT / 'results' / 'phase1' / '20261002-132157-tmall-pretrained_lora-4d00c9',
}
RUN_DIRECTORY = RUN_DIRECTORIES['ave']
SPLIT = 'valid'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_BATCH_SIZE = 32
LIMIT = None

print({
    'run_directory': str(RUN_DIRECTORY), 'split': SPLIT,
    'device': DEVICE, 'eval_batch_size': EVAL_BATCH_SIZE, 'limit': LIMIT,
})


{'run_directory': 'D:\\DataSciencePractise\\CORE\\results\\phase1\\20261005-115750-tmall-ave-9cf3dc', 'split': 'valid', 'device': 'cuda', 'eval_batch_size': 32, 'limit': None}


## 🔧 复评函数

这段代码保留原独立脚本的评估流程：核对数据与 ID 映射，恢复最佳检查点，采用共享训练器的单次全商品评价，保存分组结果与检查点来源。

每次调用创建一个新的 `results/core/evaluation-.../`。异常或中断会在新目录中记录状态；原训练目录不改动。这里只定义函数，不会开始评估。


In [3]:
def evaluate_experiment(run_directory, split='valid', device=None,
                        eval_batch_size=None, limit=None):
    if split not in {'valid', 'test'}:
        raise ValueError('split must be valid or test')
    for name, value in [('eval_batch_size', eval_batch_size), ('limit', limit)]:
        if value is not None and (isinstance(value, bool) or not isinstance(value, int) or value < 1):
            raise ValueError(f'{name} must be a positive integer')
    source_directory = Path(run_directory).resolve()
    model, config, splits = load_experiment(source_directory, device=device)
    checkpoint_path = source_directory / 'best.pth'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    best_epoch = int(checkpoint['epoch'])
    del checkpoint
    original_loader = splits[1 if split == 'valid' else 2]
    original_count = len(original_loader.dataset)
    selected_count = min(limit, original_count) if limit is not None else original_count
    if eval_batch_size is not None:
        config['eval_batch_size'] = eval_batch_size
    # Rebuild to honor optional evaluation batch size / subset. The vocabulary
    # and full candidate table are retained by Dataset.copy.
    dataset = original_loader.dataset
    if limit is not None:
        dataset = dataset.copy(dataset.inter_feat[:selected_count])
    loader = get_dataloader(config, split)(config, dataset, None, shuffle=False)
    stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    output_directory = ROOT / 'results' / 'core' / f'evaluation-{stamp}-{config["dataset"]}-{uuid4().hex[:6]}'
    output_directory.mkdir(parents=True)
    config['checkpoint_dir'] = str(output_directory)
    init_logger(config)
    summary = {
        'status': 'running', 'run_kind': 'checkpoint_evaluation',
        'source_run_directory': str(source_directory),
        'checkpoint': str(checkpoint_path), 'checkpoint_sha256': _file_hash(checkpoint_path),
        'best_epoch': best_epoch, 'model': config['model'],
        'split': split, 'device': str(config['device']),
        'eval_batch_size': config['eval_batch_size'],
        'scope': 'prefix_subset' if limit is not None else 'full',
        'limit': limit, 'original_split_count': original_count,
        'evaluated_count': selected_count, 'result_directory': str(output_directory),
        'started_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    _write_json(output_directory / 'metrics.json', summary)
    print(f'CORE_EVALUATION_DIR={output_directory}', flush=True)
    trainer = None
    try:
        trainer = CORETrainer(config, model)
        metrics = trainer.evaluate(loader, load_best_model=False, show_progress=False)
        groups = trainer.last_review_explore_result
        report = _review_explore_report(
            config['topk'], **{split: groups}, best_epoch=best_epoch, scope=summary['scope'],
        )
        report.update(source_run_directory=str(source_directory), split=split,
                      checkpoint_sha256=summary['checkpoint_sha256'],
                      original_split_count=original_count, evaluated_count=selected_count)
        _write_json(output_directory / 'review_explore.json', report)
        summary.update(status='complete', result=dict(metrics), review_explore=groups,
                       test_evaluated=split == 'test',
                       finished_at_utc=datetime.now(timezone.utc).isoformat())
        _write_json(output_directory / 'metrics.json', summary)
        print('REVIEW_EXPLORE=' + str(groups), flush=True)
        print(f'CORE_EVALUATION_COMPLETE={output_directory}', flush=True)
        return summary
    except (Exception, KeyboardInterrupt) as error:
        summary.update(status='interrupted' if isinstance(error, KeyboardInterrupt) else 'failed',
                       error=str(error), finished_at_utc=datetime.now(timezone.utc).isoformat())
        _write_json(output_directory / 'metrics.json', summary)
        raise
    finally:
        if trainer is not None:
            trainer.tensorboard.close()


## ▶️ 执行本次复评

确认上面的目录、split 和样本范围后运行此单元格。`result=None` 会先清除旧结果引用，避免评估失败后误看上一次结果。重新运行会创建新的独立结果目录，不会继续训练。


In [4]:
result = None
result = evaluate_experiment(
    run_directory=RUN_DIRECTORY,
    split=SPLIT,
    device=DEVICE,
    eval_batch_size=EVAL_BATCH_SIZE,
    limit=LIMIT,
)


D:\DataSciencePractise\CORE\.venv\Lib\site-packages\recbole\data\dataset\dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value="", inplace=True)
D:\DataSciencePractise\CORE\.venv\Lib\site-packages\recbole\data\dataset\dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

CORE_EVALUATION_DIR=D:\DataSciencePractise\CORE\results\core\evaluation-20261009-142017-tmall-7f56df
REVIEW_EXPLORE={'overall': {'count': 50579, 'fraction': 1.0, 'recall@10': 0.30823068862571423, 'mrr@10': 0.19091042395724178, 'recall@20': 0.3471005753375907, 'mrr@20': 0.19363865249139778}, 'review': {'count': 17633, 'fraction': 0.3486229462820538, 'recall@10': 0.7323767935121647, 'mrr@10': 0.50792595323163, 'recall@20': 0.7810922701752396, 'mrr@20': 0.511377583822634}, 'explore': {'count': 32946, 'fraction': 0.6513770537179462, 'recall@10': 0.08122382079766892, 'mrr@10': 0.021240818308747647, 'recall@20': 0.11482425787652523, 'mrr@20': 0.023581875457351537}}
CORE_EVALUATION_COMPLETE=D:\DataSciencePractise\CORE\results\core\evaluation-20261009-142017-tmall-7f56df


## 📊 本次分组结果

指标为 0–1 数值，不是商品点击概率。空组指标为空值，不应按 0 分解释；`fraction` 是该组占全部样本的比例。

此单元格读取刚保存的 JSON，不会再次运行模型。两组按样本数加权的未舍入指标应还原整体指标。


In [5]:
if result is None or result['status'] != 'complete':
    raise RuntimeError('请先成功运行复评单元格。')

OUTPUT_DIRECTORY = Path(result['result_directory'])
report = json.loads((OUTPUT_DIRECTORY / 'review_explore.json').read_text(encoding='utf-8'))
groups = report[result['split']]

print(f"评价范围：{result['scope']}；split：{result['split']}；样本数：{result['evaluated_count']}")
print(f'结果目录：{OUTPUT_DIRECTORY}')
for metric in ('recall@10', 'mrr@10', 'recall@20', 'mrr@20'):
    recomposed = sum(
        group['fraction'] * group[metric]
        for group in (groups['review'], groups['explore']) if group['count']
    )
    if abs(recomposed - groups['overall'][metric]) > 1e-12:
        raise AssertionError(f'分组加权校验失败：{metric}')

group_table = pd.DataFrame.from_dict(groups, orient='index')
group_table.index.name = 'group'
group_table


评价范围：full；split：valid；样本数：50579
结果目录：D:\DataSciencePractise\CORE\results\core\evaluation-20261009-142017-tmall-7f56df


,count,fraction,recall@10,mrr@10,recall@20,mrr@20
group,,,,,,
overall,50579,1.000000,0.308231,0.190910,0.347101,0.193639
review,17633,0.348623,0.732377,0.507926,0.781092,0.511378
explore,32946,0.651377,0.081224,0.021241,0.114824,0.023582


## 📊 可选：查看已有三组验证结果

下面只读取 2026-10-09 已保存的完整验证集报告，不会加载模型或启动新复评。它不替代上面的本次运行，也不包含测试集结果。

三组必须来自相同验证集范围且样本数一致。当前比较仍为单 seed；explore 的微小优势不能视为稳定收益或语言预训练贡献证明。


In [6]:
SAVED_REPORTS = {
    'AVE': ROOT / 'results' / 'core' / 'evaluation-20261009-112459-tmall-ebccc6' / 'review_explore.json',
    'TRM': ROOT / 'results' / 'core' / 'evaluation-20261009-112630-tmall-d16aee' / 'review_explore.json',
    'LLM': ROOT / 'results' / 'core' / 'evaluation-20261009-112657-tmall-e1ee18' / 'review_explore.json',
}
comparison_rows = []
for name, report_path in SAVED_REPORTS.items():
    if not report_path.is_file():
        print(f'跳过不存在的历史报告：{report_path}')
        continue
    saved_report = json.loads(report_path.read_text(encoding='utf-8'))
    if saved_report['scope'] != 'full' or saved_report.get('split') != 'valid':
        raise ValueError(f'不是完整验证集报告：{report_path}')
    saved_groups = saved_report['valid']
    comparison_rows.append({
        'model': name,
        'review_count': saved_groups['review']['count'],
        'explore_count': saved_groups['explore']['count'],
        'review_mrr@20': saved_groups['review']['mrr@20'],
        'explore_mrr@20': saved_groups['explore']['mrr@20'],
    })
comparison_table = pd.DataFrame(comparison_rows)
if not comparison_table.empty:
    if comparison_table[['review_count', 'explore_count']].nunique().gt(1).any():
        raise ValueError('历史报告的样本范围不一致，不能直接比较。')
    comparison_table = comparison_table.set_index('model')
comparison_table


,review_count,explore_count,review_mrr@20,explore_mrr@20
model,,,,
AVE,17633,32946,0.511378,0.023582
TRM,17633,32946,0.511815,0.023324
LLM,17633,32946,0.499573,0.023754
